# Debugging daemonflux with the open data


Dataset must be obtained from the PRD 2023 paper of IceCube

https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/B4RITM&version=1.0

## Setup

Usually you would need to export the location of the data, and the config files to `PISA_RESOURCES`. For this example however, the data is inside our package.

In [21]:
import numpy as np
from uncertainties import unumpy as unp
import matplotlib.pyplot as plt
import copy
import os
import pisa
from pisa.core.distribution_maker import DistributionMaker
from pisa.core.pipeline import Pipeline
from pisa.analysis.analysis import Analysis
from pisa import FTYPE, ureg

In [22]:
fridge_path = '/home/jp/icecube/wg-oscillations-fridge/analysis'

In [23]:
os.environ['PISA_RESOURCES'] = fridge_path

In [24]:
myfiles_path = '/home/jp/daemonflux/oscillations_release_2023'

In [25]:
## Model

We can now instantiate the `model` (given our configs) that we later fit to data. This now containes two `Pipelines` in a `DistributionMaker`, one for our neutrinos, and one for the background muons.

In [51]:
model = DistributionMaker([
    os.path.join(myfiles_path,"pisa_settings/pipeline/neutrino_pipeline_daemon_simple.cfg"),
    #os.path.join(myfiles_path,"pisa_settings/pipeline/muon_pipeline.cfg")
])

[ WARNING] /home/jp/anaconda3/envs/pisa/lib/python3.10/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")



In [52]:
model.pipelines[0]

stage number,name,calc_mode,apply_mode,has setup,has compute,has apply,# fixed params,# free params
0,csv_loader,events,events,True,False,True,0,0
1,daemon_flux,events,events,True,True,False,2,24
2,two_nu_osc,events,events,False,False,True,2,0


In [54]:
model.theta23 = 0.
model.get_outputs()

KeyError: 'Data errors not present in Container'

In [38]:
## Verifying that there are things loaded
#model.pipelines[0].data['numu_cc']['true_energy']
#model.pipelines[1].data['icc']['reco_energy']
#model.pipelines[2].data['total']['reco_energy']

In [40]:
len(model.pipelines[0].data['numu_cc']['true_energy'])

185464

In [42]:
model.pipelines[0].data['numu_cc']

Container containing keys ['initial_weights', 'weights', 'true_energy', 'true_coszen', 'pdg_code', 'interaction', 'weighted_aeff', 'reco_energy', 'reco_coszen', 'pid', 'linear_fit_maccqe', 'quad_fit_maccqe', 'linear_fit_maccres', 'quad_fit_maccres', 'bjorken_y', 'dis', 'nu_flux']

In [46]:
mynans = np.isnan(model.pipelines[0].data['numu_cc']['nu_flux'][:,0])

In [47]:
model.pipelines[0].data['numu_cc']['true_energy'][mynans]

array([ 25.13912901,  22.92467667,  67.4029702 , ...,  84.79055654,
        92.31537452, 145.23919077])

In [49]:
model.get_outputs()

[ WARNING] /home/jp/software/pisa/pisa/stages/flux/daemon_flux.py:166: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "make_2d_flux_map" failed type inference due to: non-precise type pyobject
During: typing of argument at /home/jp/software/pisa/pisa/stages/flux/daemon_flux.py (181)

File "../pisa/stages/flux/daemon_flux.py", line 181:
def make_2d_flux_map(flux_obj,
    <source elided>

    for index in range(len(icangles)):
    ^

  @jit(forceobj=True)



KeyError: 'Data errors not present in Container'